##### Installing dependencies

In [1]:
%%capture
# System dependency for python-magic
!apt-get install -y libmagic1 -q

# Core ML stack — let Unsloth control transformers version
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q
!pip install --no-deps xformers peft bitsandbytes -q
!pip install "trl>=0.13.0" "accelerate>=1.0.0" -q

# HuggingFace
!pip install transformers torch huggingface_hub datasets sentence-transformers -q

# LangChain full stack
!pip install "langchain>=0.2.0,<0.3.0" \
             "langchain-community>=0.2.0,<0.3.0" \
             langchain-core \
             langchain-text-splitters -q

# PDF and document processing
!pip install pymupdf "unstructured[pdf]" python-magic pypdf -q

# Vector store
!pip install faiss-cpu -q

# Utilities
!pip install rich -q

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, "-m", "pip", "show", "unsloth"],
    capture_output=True, text=True
)
print(result.stdout if result.stdout else "❌ unsloth is NOT installed")
print(result.stderr if result.stderr else "")

Name: unsloth
Version: 2026.6.1
Summary: 2-5X faster training, reinforcement learning & finetuning
Home-page: https://unsloth.ai
Author: Unsloth AI team
Author-email: info@unsloth.ai
License: 
Location: /usr/local/lib/python3.12/dist-packages
Requires: nest-asyncio, pydantic, pyyaml, typer
Required-by: 




##### Necessary imports

In [4]:
import torch
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported
from datasets import load_dataset


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [5]:
!pip install langchain-text-splitters

In [6]:
!pip install langchain-core

In [7]:
import unsloth, transformers, trl, langchain
print("unsloth:", unsloth.__version__)
print("transformers:", transformers.__version__)
print("trl:", trl.__version__)
print("langchain:", langchain.__version__)

unsloth: 2026.6.1
transformers: 5.5.0
trl: 0.24.0
langchain: 0.2.17


In [8]:
import logging
import numpy as np
import faiss
import os
import pandas as pd
import magic
import nltk

from langchain_community.document_loaders import TextLoader
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.document_loaders import UnstructuredURLLoader
from langchain_community.document_loaders import DirectoryLoader  # FIX: moved to langchain_community

# FIX: All of the following were moved from langchain.* to langchain_community.* in langchain>=0.2.0
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores import Chroma

# FIX: RecursiveCharacterTextSplitter and PromptTemplate remain in langchain core
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import PromptTemplate  # FIX: removed duplicate 'from langchain import PromptTemplate'

# google.colab imports are Colab-only — safe here since this notebook targets Colab
from google.colab import userdata

from transformers import TextStreamer
from rich.console import Console
from rich.markdown import Markdown
from rich.table import Table
from rich.panel import Panel
from rich.text import Text


##### Mount Drive

In [9]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
import transformers; print(transformers.__version__)

5.5.0


##### Base model loading in quantized in 4-bit

In [11]:
max_seq_length = 4096  # You can set this upto {8192 - (max number of output tokens you want)}
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)


==((====))==  Unsloth 2026.6.1: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/198 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.25k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.6k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

Unsloth: Will load unsloth/llama-3-8b-bnb-4bit as a legacy tokenizer.


##### Initializing Model for Low Rank Adaptation

In [12]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = True,
    loftq_config = None,
)


Unsloth 2026.6.1 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [13]:
#@title ##### System Prompt — used for RAG with the fine-tuned LLM
alpaca_prompt = """Below is a query asked by a user , paired with additonal data that provides further context. Write a response that appropriately answers the query.
You must answer only from the given data and not make up anything. Be as detailed as possible. If the answer is not present in the data, just say "I don't know".

### Query:
{}

### Data:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token


In [14]:
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN  # EOS token as stop signal
        texts.append(text)
    return { "text" : texts }


In [15]:
#@title ##### Loading the dataset
dataset = load_dataset("yahma/alpaca-cleaned", split = "train")
dataset = dataset.map(formatting_prompts_func, batched = True)


README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

alpaca_data_cleaned.json:   0%|          | 0.00/44.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/51760 [00:00<?, ? examples/s]

Map:   0%|          | 0/51760 [00:00<?, ? examples/s]

In [16]:
dataset


Dataset({
    features: ['output', 'input', 'instruction', 'text'],
    num_rows: 51760
})

In [17]:
import trl; print(trl.__version__)

0.24.0


In [18]:
import transformers; print(transformers.__version__)

5.5.0


##### Initializing training parameters using LoRA

In [22]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = max_seq_length,
        dataset_num_proc = 1,        # ← this was the only problem
        packing = False,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 100,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=1):   0%|          | 0/51760 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 51,760 | Num Epochs = 1 | Total steps = 100
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,0.586793
2,0.530353
3,0.669796
4,0.704143
5,0.632691
6,0.808550
7,0.540701
8,0.674221
9,0.517018
10,0.450223


In [23]:
#@title ##### Memory statistics
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")


GPU = Tesla T4. Max memory = 14.563 GB.
11.549 GB of memory reserved.


##### LoRA enabled Fine-Tuning

In [24]:
trainer_stats = trainer.train()


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 51,760 | Num Epochs = 1 | Total steps = 100
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,0.299076
2,0.223656
3,0.298169
4,0.337675
5,0.244356
6,0.379578
7,0.273233
8,0.328085
9,0.283155
10,0.262503


In [ ]:
#@title ##### Final memory and time statistics after training
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")


In [ ]:
#@title ##### Test Inference
FastLanguageModel.for_inference(model)
inputs = tokenizer(
[
    alpaca_prompt.format(
        "Say the opposite of the data provided.",  # query
        "Hello",                                    # data
        "",                                         # output — leave blank
    )
], return_tensors = "pt").to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128)


##### Save the Model (saves only the LoRA adapters for efficiency)

In [ ]:
model.save_pretrained("/content/drive/MyDrive/aries_llama3_8b/aries_model")
tokenizer.save_pretrained("/content/drive/MyDrive/aries_llama3_8b/aries_model")
# model.push_to_hub("your_name/lora_model", token = "...")      # Online saving
# tokenizer.push_to_hub("your_name/lora_model", token = "...")  # Online saving


##### Load the Model from saved path

In [ ]:
max_seq_length = 4096
dtype = None
load_in_4bit = True

if True:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "/content/drive/MyDrive/aries_llama3_8b/aries_model",
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
    )
    FastLanguageModel.for_inference(model)


In [ ]:
#@title ##### System Prompt (inference)
alpaca_prompt = """Below is a query asked by a user , paired with additonal data that provides further context. Write a response that appropriately answers the query.
You must answer only from the given data and not make up anything. Be as detailed as possible. If the answer is not present in the data, just say "I don't know".

### Query:
{}

### Data:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token


In [ ]:
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts }


##### Document Loading and Preprocessing

In [ ]:
documents = []
file_path = '/content/drive/MyDrive/mml-book.pdf'  # Path to your document
loader = PyMuPDFLoader(file_path)
documents = loader.load()

# To load a full directory of PDFs instead, uncomment below:
# pdf_directory = "/content/drive/MyDrive/"
# documents = []
# for root, dirs, files in os.walk(pdf_directory):
#     for file in files:
#         if file.endswith(".pdf"):
#             file_path = os.path.join(root, file)
#             loader = PyMuPDFLoader(file_path)
#             documents.extend(loader.load())


In [ ]:
print(f"Number of documents loaded: {len(documents)}")


In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500)
split_documents = text_splitter.split_documents(documents)
print(f"Number of chunks: {len(split_documents)}")


In [ ]:
#@title ##### Embedding function — converts text chunks into vectors
# HuggingFaceEmbeddings defaults to 'sentence-transformers/all-mpnet-base-v2'
# FIX: imported from langchain_community.embeddings (not deprecated langchain.embeddings)
embeddings = HuggingFaceEmbeddings()


##### Creation of FAISS vector database

In [ ]:
save_directory_faiss = "/content/drive/MyDrive/FAISS_aries"


In [ ]:
#@title ##### Create and save the FAISS vector database
vector_db_faiss = FAISS.from_documents(documents = split_documents, embedding = embeddings)
vector_db_faiss.save_local(save_directory_faiss)


In [ ]:
#@title ##### Or load an existing FAISS database
vector_db_faiss = FAISS.load_local(save_directory_faiss, embeddings, allow_dangerous_deserialization=True)


##### Searching the vector database (demonstration)

In [ ]:
user_input = input("Please enter your query: ")


In [ ]:
# similarity_search_with_score returns (Document, float) pairs
docs = vector_db_faiss.similarity_search_with_score(user_input, k=6)


In [ ]:
# docs


In [ ]:
similar_documents = []
scores = []
for doc, score in docs:
    similar_documents.append(doc)
    scores.append(score)


In [ ]:
# similar_documents


In [ ]:
page_contents = [doc.page_content for doc in similar_documents]
metadata = [doc.metadata for doc in similar_documents]

print("Page Contents:")
for i, content in enumerate(page_contents):
    print(f"Document {i+1} Content:\n{content}\n")

print("Metadata:")
for i, meta in enumerate(metadata):
    print(f"Document {i+1} Metadata:\n{meta}\n")


In [ ]:
# page_contents


##### Inference and response by the model

In [ ]:
def extract_response(text):
    response_marker = "### Response:"
    start_index = text.find(response_marker)
    if start_index == -1:
        return "Response marker not found"
    start_index += len(response_marker)
    return text[start_index:].strip()

def save_conversation_log(conversation_log, filename="conversation_log.txt"):
    with open(filename, "w") as file:
        for entry in conversation_log:
            file.write(entry + "\n")
            file.write("\n" + "-"*80 + "\n")


In [ ]:
# Initialize conversation log and rich console
conversation_log = []
console = Console()

console.print("[bold green]Welcome! Type your query below. Type 'exit' to stop.[/bold green]")

while True:
    user_input = input("Please enter your query: ")

    if user_input.lower() == "exit":
        break

    conversation_log.append(f"User: {user_input}")

    docs = vector_db_faiss.similarity_search_with_score(user_input, k=6)

    similar_documents = []
    scores = []
    for doc, score in docs:
        similar_documents.append(doc)
        scores.append(score)

    page_contents = [doc.page_content for doc in similar_documents]
    metadata = [doc.metadata for doc in similar_documents]

    # Display retrieved docs in a rich table
    table = Table(title="Retrieved Documents")
    table.add_column("Page Content", justify="left", style="cyan", no_wrap=False)
    table.add_column("Metadata", justify="left", style="magenta")
    table.add_column("Score", justify="left", style="green")
    for content, meta, score in zip(page_contents, metadata, scores):
        table.add_row(content, str(meta), str(score))
    console.print(table)

    # Prepare model inputs
    inputs = tokenizer(
        [
            alpaca_prompt.format(
                user_input,    # query
                page_contents, # context data
                "",            # output — leave blank
            )
        ],
        return_tensors="pt"
    ).to("cuda")

    # Generate response
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        pad_token_id=tokenizer.eos_token_id,
        use_cache=True
    )
    response = extract_response(tokenizer.decode(outputs[0], skip_special_tokens=True))

    conversation_log.append(f"Model: {response}")

    # Display using rich panels
    console.print(Panel(Text(f"Query:\n\n{user_input}", justify="left"), title="User Query"))
    console.print(Panel(Text(f"Response:\n\n{response}", justify="left"), title="Model Response"))

    # Log retrieved docs
    conversation_log.append("Retrieved Documents:\n")
    for i, (content, meta, score) in enumerate(zip(page_contents, metadata, scores)):
        conversation_log.append(f"Document {i+1} Metadata:\n{meta}")
        conversation_log.append(f"Document {i+1} Content:\n{content}")
        conversation_log.append(f"Document {i+1} Score:\n{score}")
        conversation_log.append("\n" + "-"*80 + "\n")

save_conversation_log(conversation_log)
console.print("[bold green]Conversation saved to conversation_log.txt[/bold green]")
